# VWAP Events 01 — does a VWAP touch precede a violent move?

Pre-registration: `discussions/VWAP_Events/001_claude_branch_preregistration.md`. Parameters, windows and gates are fixed in
`src/data_engineering/modeling/vwap_events.py` (`registration()` below).

- **Data:** Binance USD-M perpetual 1-minute klines (`tools/download_binance_um_klines.py`); VWAP $=\sum \text{quote vol}/\sum \text{vol}$ since the anchor (exact).
- **Anchors:** session VWAP anchored at 08:00 UTC (Deribit expiry) and 4-hour VWAP (00/04/…/20 UTC).
- **Event:** the bar's range contains the VWAP known at its open, after 15 closes on one side with at least one $\ge 1\,\hat\sigma\sqrt{15}$ away.
- **Outcome (primary):** $y=\log R_{15}-\text{median}_{\text{hour}}\log R_{15}$, with $R_k=\sqrt{\sum_{i=1}^{k} r_{t+i}^2}\,/\,(\hat\sigma_{t-1}\sqrt{k})$.
- **Placebos:** the same VWAP shifted by a constant offset per window; a VWAP of the same period anchored at a random phase.
- **Decision (test window from 2026-04-01):** per anchor, $\Delta=\bar y_{\text{real}}-\bar y_{\text{placebo}}$ must have CI lower bound $>0$ (Bonferroni over 4 comparisons) **and** $e^{\Delta}\ge 1.10$, against both placebos.

Environment: `VE01_ASSETS` (comma list), `VE01_N_PLACEBO` (default 20), `BINANCE_UM_DIR` (default `<parent of DERIBIT_DATA_ROOT>/Binance`), `VE01_SYNTHETIC=1` (synthetic smoke run, outputs to a temp dir).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import vwap_events as ve

SYNTHETIC = os.getenv('VE01_SYNTHETIC', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('VE01_ASSETS', ','.join(ve.ASSETS)).split(',') if a.strip()]
P = replace(ve.Params(), n_placebo=int(os.getenv('VE01_N_PLACEBO', str(ve.Params().n_placebo))))
if SYNTHETIC:
    BIN_DIR = None
    OUT = Path(tempfile.mkdtemp(prefix='ve01_'))
else:
    BIN_DIR = Path(os.getenv('BINANCE_UM_DIR') or Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'VWAP_Events' / '01'
OUT.mkdir(parents=True, exist_ok=True)
REG = ve.registration(P)
print('VWAP Events 01 |', ve.VERSION, '| synthetic' if SYNTHETIC else f'| data {BIN_DIR}', '| out', OUT)
print(json.dumps({k: REG[k] for k in ('m_approach', 'd_approach', 'cooldown', 'horizons', 'primary_h', 'n_placebo',
                                       'ci_level', 'min_effect', 'test_start')}, default=str))

## 1. Data and events (real touches and placebo touches)

In [ ]:
events, coverage = [], []
for i, a in enumerate(ASSETS):
    t0 = time.time()
    if SYNTHETIC:
        df = ve.synthetic_klines(n_days=180, seed=10 + i, start='2025-12-01')
    else:
        path = BIN_DIR / f'{a}USDT_um_1m.parquet'
        if not path.exists():
            print(f'{a}: no Binance file at {path} -> skipped'); continue
        df = pd.read_parquet(path, columns=['open_time', 'high', 'low', 'close', 'volume', 'quote_volume'])
    ev, info = ve.run_asset(df, a, P)
    events.append(ev); coverage.append(info)
    print(f'{a}: {info["bars"]:,} bars, {len(ev):,} event rows, {time.time() - t0:.0f}s')
EV = pd.concat(events, ignore_index=True)
coverage = pd.DataFrame(coverage); display(coverage)
counts = EV.groupby(['anchor', 'window', 'kind', 'asset']).size().unstack('asset').fillna(0).astype(int)
display(counts)

## 2. Primary readout — $y_{15}$, real vs each placebo, pooled assets

`ratio` $=e^{\Delta}$: how much larger the time-of-day-normalised 15-minute realised volatility is after a real touch than after a placebo touch. CI at the Bonferroni level; bootstrap over UTC days.

In [ ]:
primary = ve.readout(EV, P)
display(primary.round(4))
dec = ve.decision(primary)
print('DECISION (test window):'); display(dec)

## 3. Secondary — horizons, absolute move, per asset, direction (descriptive, no gates)

In [ ]:
rows = []
for k in P.horizons:
    for col, meas in ((f'y{k}', 'R (realised vol)'), (f'logM{k}', 'M (absolute move)')):
        r = ve.readout(EV, P, col=col); r['k'] = k; r['measure'] = meas; rows.append(r)
horizons = pd.concat(rows, ignore_index=True)
display(horizons[horizons['window'] == 'test'].pivot_table(index=['anchor', 'placebo', 'k'], columns='measure',
                                                            values='ratio', aggfunc='first').round(3))

by_asset = ve.readout(EV[EV['window'] == 'test'], P, by=('anchor', 'asset'))
display(by_asset[['anchor', 'asset', 'placebo', 'n_real', 'ratio', 'lo', 'hi']].round(4))

direction = ve.readout(EV, P, col=f'cross{P.primary_h}')
direction = direction.rename(columns={'mean_real': 'p_cross_real', 'mean_placebo': 'p_cross_placebo'})
display(direction[['anchor', 'window', 'placebo', 'n_real', 'p_cross_real', 'p_cross_placebo', 'delta', 'lo', 'hi']].round(4))

In [ ]:
fig, axes = plt.subplots(1, len(P.anchors), figsize=(12, 4), sharey=True)
axes = np.atleast_1d(axes)
t = EV[EV['window'] == 'test']
for ax, an in zip(axes, [a.name for a in P.anchors]):
    for kind, st in (('real', '-o'), ('offset', '--s'), ('phase', ':^')):
        g = t[(t['anchor'] == an) & (t['kind'] == kind)]
        ax.plot(P.horizons, [np.exp(g[f'y{k}'].mean()) for k in P.horizons], st, label=kind)
    ax.axhline(1, color='grey', lw=0.8); ax.set_title(f'{an} — test window'); ax.set_xlabel('horizon k (min)')
axes[0].set_ylabel('exp(mean y): normalised realised vol'); axes[0].legend()
plt.tight_layout(); plt.show()

## 4. Save compact outputs

In [ ]:
coverage.to_csv(OUT / '01_coverage.csv', index=False)
counts.reset_index().to_csv(OUT / '01_event_counts.csv', index=False)
primary.to_csv(OUT / '01_primary_readout.csv', index=False)
dec.to_csv(OUT / '01_decision.csv', index=False)
horizons.to_csv(OUT / '01_horizons.csv', index=False)
by_asset.to_csv(OUT / '01_by_asset_test.csv', index=False)
direction.to_csv(OUT / '01_direction.csv', index=False)
manifest = {'registration': REG, 'assets_run': list(coverage['asset']), 'synthetic': SYNTHETIC,
            'decision': dec.to_dict(orient='records'), 'run_utc': str(pd.Timestamp.now(tz='UTC'))}
(OUT / '01_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print('saved to', OUT, sorted(p.name for p in OUT.iterdir()))